# OliveSmart — OLV-5 : Prévision du rendement des oliveraies

## 1. Problématique

L'objectif de ce projet est de prédire le rendement des oliveraies marocaines avant la récolte, à partir des données historiques, climatiques et satellitaires.

Il s'agit d'un problème de régression supervisée.

- Variables d'entrée (X) : données climatiques, historiques et caractéristiques des parcelles disponibles avant la récolte.
- Variable cible (y) : rendement réel en kilogrammes par hectare (kg/ha).

## 2. Objectif expérimental

Comparer au moins quatre algorithmes, dont une baseline simple, afin de sélectionner le modèle le plus adapté selon ses performances et son coût de calcul.

Modèles envisagés : baseline historique, Ridge, Random Forest, XGBoost, MLP et éventuellement LSTM léger selon la disponibilité des séries temporelles.

## 3. État d'avancement

- Problématique identifiée.
- Modèles de comparaison identifiés.
- Vérification des datasets : en cours.
- Entraînement : non commencé.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

print("OliveSmart OLV-5")
print("Environnement Python prêt")
print("Version Pandas :", pd.__version__)

OliveSmart OLV-5
Environnement Python prêt
Version Pandas : 3.0.3


## 4. Identification des sources de données

Notre projet nécessite plusieurs types de données pour prédire le rendement des oliveraies.

| Source | Données recherchées | État |
|---|---|---|
| FAOSTAT | Production et rendement historique au niveau national | À vérifier |
| NASA POWER | Température, précipitations et climat | À vérifier |
| Sentinel-2 | Indices de végétation | À vérifier |
| FAO WaPOR | Évapotranspiration et données hydriques | À vérifier |
| Collecte Maroc | Rendements réels par parcelle et campagne | À obtenir / constituer |

**Variable cible (y) :** rendement réel en kg/ha.

**Unité d'observation :** une parcelle pendant une campagne agricole.

**Point à clarifier avec le professeur :** disponibilité des données locales de récolte nécessaires à l'entraînement supervisé.

In [8]:
# Dossier actuel de Jupyter
print("Dossier actuel :", Path.cwd())
# Rechercher tous les fichiers CSV
fichiers_csv = list(Path(".").glob("*.csv"))
print("Listes des fichiers CSV trouvés / Notre DataSets :")
for fichier in fichiers_csv:
    print(fichier.name)

Dossier actuel : C:\Users\Ultrapc\Documents\Dev\5_YearEmsi\JEE_DEEPLEARNING_PRJ
Listes des fichiers CSV trouvés / Notre DataSets :
FAOSTAT_data_en_11-2-2026-1y.csv
FAOSTAT_data_en_11-2-2026_3y.csv


## Chargement et affichage des dataset

In [12]:
# Charger le premier dataset : Yield uniquement
df_1y = pd.read_csv("FAOSTAT_data_en_11-2-2026-1y.csv")

# Charger le deuxième dataset : 3 éléments
df_3y = pd.read_csv("FAOSTAT_data_en_11-2-2026_3y.csv")

# Afficher les dimensions
print("Yield: Étudier le rendement historique des olives.")
print("Production Quantity: Étudier la quantité totale produite au Maroc.")
print("Area harvested:Étudier l'évolution de la superficie récoltée.\n")

print("DATASET 1 : Yield uniquement")
print("Dimensions :", df_1y.shape)
print("Colonnes :", df_1y.columns.tolist())
display(df_1y.head())

print("\nDATASET 2 : 3 éléments: Yield, Production Quantity, Area harvested")
print("Dimensions :", df_3y.shape)
print("Colonnes :", df_3y.columns.tolist())
display(df_3y.head())

Yield: Étudier le rendement historique des olives.
Production Quantity: Étudier la quantité totale produite au Maroc.
Area harvested:Étudier l'évolution de la superficie récoltée.

DATASET 1 : Yield uniquement
Dimensions : (64, 15)
Colonnes : ['Domain Code', 'Domain', 'Area Code (M49)', 'Area', 'Element Code', 'Element', 'Item Code (CPC)', 'Item', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Flag Description', 'Note']


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,504,Morocco,5412,Yield,1450,Olives,1961,1961,kg/ha,900.6,E,Estimated value,NaN
1,QCL,Crops and livestock products,504,Morocco,5412,Yield,1450,Olives,1962,1962,kg/ha,906.3,E,Estimated value,NaN
2,QCL,Crops and livestock products,504,Morocco,5412,Yield,1450,Olives,1963,1963,kg/ha,869.6,E,Estimated value,NaN
3,QCL,Crops and livestock products,504,Morocco,5412,Yield,1450,Olives,1964,1964,kg/ha,885.0,E,Estimated value,NaN
4,QCL,Crops and livestock products,504,Morocco,5412,Yield,1450,Olives,1965,1965,kg/ha,890.3,E,Estimated value,NaN



DATASET 2 : 3 éléments: Yield, Production Quantity, Area harvested
Dimensions : (192, 15)
Colonnes : ['Domain Code', 'Domain', 'Area Code (M49)', 'Area', 'Element Code', 'Element', 'Item Code (CPC)', 'Item', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Flag Description', 'Note']


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,504,Morocco,5312,Area harvested,1450,Olives,1961,1961,ha,155000.0,E,Estimated value,NaN
1,QCL,Crops and livestock products,504,Morocco,5412,Yield,1450,Olives,1961,1961,kg/ha,900.6,E,Estimated value,NaN
2,QCL,Crops and livestock products,504,Morocco,5510,Production,1450,Olives,1961,1961,t,139600.0,A,Official value,NaN
3,QCL,Crops and livestock products,504,Morocco,5312,Area harvested,1450,Olives,1962,1962,ha,160000.0,E,Estimated value,NaN
4,QCL,Crops and livestock products,504,Morocco,5412,Yield,1450,Olives,1962,1962,kg/ha,906.3,E,Estimated value,NaN
